![digitizing_team](digitizing_team.png)


DigiNsure Inc. is an innovative insurance company focused on enhancing the efficiency of processing claims and customer service interactions. Their newest initiative is digitizing all historical insurance claim documents, which includes improving the labeling of some IDs scanned from paper documents and identifying them as primary or secondary IDs.

To help them in their effort, you'll be using multi-modal learning to train an Optical Character Recognition (OCR) model. To improve the classification, the model will use **images** of the scanned documents as input and their **insurance type** (home, life, auto, health, or other). Integrating different data modalities (such as image and text) enables the model to perform better in complex scenarios, helping to capture more nuanced information. The **labels** that the model will be trained to identify are of two types: a primary and a secondary ID, for each image-insurance type pair.

In [13]:
import torch
import torch.nn as nn

class OCRModel(nn.Module):
    def __init__(self):
        super(OCRModel, self).__init__()
        
        # Image processing layers
        self.image_layer = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),  
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # (B, 16, 32, 32)
            nn.Flatten(),  # (B, 16*32*32)
            nn.Linear(16 * 32 * 32, 128),  # (B, 128)
            nn.ReLU()
        )
        
        # Type (insurance type) processing layers
        self.type_layer = nn.Sequential(
            nn.Linear(5, 10),  
            nn.ReLU()
        )
        
        # Classifier layers
        self.classifier = nn.Sequential(
            nn.Linear(128 + 10, 64)  
        )
        
    def forward(self, x_image, x_type):
        img_feat = self.image_layer(x_image)  
        type_feat = self.type_layer(x_type)   
        x = torch.cat((img_feat, type_feat), dim=1)  
        out = self.classifier(x)  
        return out

In [14]:
from torch.utils.data import DataLoader
import torch.optim as optim
from project_utils import ProjectDataset

# 1. Instantiate the dataset from project_utils
dataset = ProjectDataset(num_samples=100)

# 2. Create the conveyor belt (DataLoader)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)

# 3. Instantiate the model
model = OCRModel()

# 4. Define the optimizer using Adam
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 5. Define the cross-entropy loss function
criterion = nn.CrossEntropyLoss()

In [15]:
num_epochs = 10 
model.train()

for epoch in range(num_epochs):
    running_loss = 0.0
    
    # Loop through batches provided by your new train_loader
    for inputs, labels in train_loader:
        # inputs contains both the image tensor and the type vector
        images, types = inputs
        
        # 1. Clear gradients
        optimizer.zero_grad()
        
        # 2. Forward pass
        outputs = model(images, types)
        
        # 3. Calculate loss
        loss = criterion(outputs, labels)
        
        # 4. Backward pass
        loss.backward()
        
        # 5. Update weights
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {epoch_loss:.4f}")

Epoch 1/10 - Loss: 1.6115
Epoch 2/10 - Loss: 0.8280
Epoch 3/10 - Loss: 0.7111
Epoch 4/10 - Loss: 0.7193
Epoch 5/10 - Loss: 0.7856
Epoch 6/10 - Loss: 0.6953
Epoch 7/10 - Loss: 0.5280
Epoch 8/10 - Loss: 0.4623
Epoch 9/10 - Loss: 0.3946
Epoch 10/10 - Loss: 0.4224
